# PoolPy Decoder - User-Friendly Interface

This notebook provides an easy-to-use interface for decoding readouts using the PoolPy decoder.

## Instructions:
1. **Cell 1**: Install dependencies from requirements.txt
2. **Cell 2**: Import required libraries
3. **Cell 3**: Define your decoder configuration (paths, parameters)
4. **Cell 4**: Run the decoder
5. **Cell 5**: View and download results

Simply fill in the configuration variables in Cell 3 and run the cells in order.


In [ ]:
import subprocess
import sys
import os

# Setup environment from requirements.txt
print("Environment Setup")
print("=" * 70)

# Get the working directory
working_dir = os.getcwd()
print(f"Working directory: {working_dir}")
print()

requirements_path = os.path.join(working_dir, "requirements.txt")

# Try to read requirements.txt to show what packages are needed
try:
    with open(requirements_path, 'r') as f:
        requirements = [line.strip() for line in f if line.strip() and not line.startswith('#')]
    
    print(f"✓ Requirements file found: {requirements_path}")
    print(f"\nRequired packages ({len(requirements)} total):")
    for req in requirements:
        print(f"  - {req}")
    
    print("\n" + "-" * 70)
    print("Environment Status:")
    print(f"  Python version: {sys.version}")
    print(f"  Python executable: {sys.executable}")
    
    # Check if key packages are available
    key_packages = ['pandas', 'numpy', 'scipy', 'sklearn']
    for pkg_name in key_packages:
        try:
            __import__(pkg_name)
            print(f"  ✓ {pkg_name} is installed")
        except ImportError:
            print(f"  ✗ {pkg_name} is NOT installed")
    
except FileNotFoundError:
    print(f"✗ Requirements file not found: {requirements_path}")
    print("Please ensure requirements.txt exists in the working directory.")

print("=" * 70)
print("\nNote: For UV environments, dependencies are typically pre-installed.")
print("If you need to install packages, use: uv pip install -r requirements.txt")


In [ ]:
import pandas as pd
import numpy as np
import subprocess
import os
from pathlib import Path

print("✓ All libraries imported successfully!")
print(f"\nPython executable: {sys.executable}")
print(f"Working directory: {os.getcwd()}")


## Configuration

Edit the variables below to specify your decoder settings. By default, the notebook now uses the test folder in this workspace.

Parameters:
- path_to_WA: Path to the well assignment (WA) matrix CSV file (defaults to working_dir/test/test_wa_notebook.csv)
- readout: Path to the readout CSV file, binary or continuous (defaults to working_dir/test/test_continuous_notebook.csv)
- differentiate: Maximum number of expected active compounds (for sparsity control), default: 2
- diluting: Whether to use dilution scaling in continuous decoding (True/False), default: True
- min_signal: Optional minimum signal threshold (set to None to skip), default: None
- output_dir: Directory where decoded results will be saved (defaults to working_dir/test)
- python_executable: Python executable to use (automatically uses the current Python kernel)

To use custom paths:
Simply edit the lines in Cell 5, for example:
```python
path_to_WA = os.path.join(working_dir, "path/to/my_wa_matrix.csv")
readout = os.path.join(working_dir, "data/my_readout.csv")
```
The decoder auto-infers if the readout is binary or continous.

In [ ]:
# ============================================================================
# USER CONFIGURATION - EDIT THESE VARIABLES
# ============================================================================

# Get the working directory (set automatically)
working_dir = os.getcwd()

# Default test folder containing decoder and sample files
test_dir = os.path.join(working_dir, "test")

# Path to the well assignment (WA) matrix CSV file
path_to_WA = os.path.join(test_dir, "test_wa_notebook.csv")

# Path to the readout CSV file (binary or continuous format)
readout = os.path.join(test_dir, "test_continuous_notebook.csv")

# Maximum number of expected postive samples
differentiate = 2

# Whether to use dilution scaling in continuous decoding (True/False)
diluting = True

# Optional minimum signal threshold (set to None to skip)
min_signal = None

# Output directory for results (defaults to test folder)
output_dir = test_dir

# Python executable (uses the current Python that's running the notebook)
python_executable = sys.executable

# ============================================================================
# Validation: Check if input files exist
# ============================================================================

print("Configuration:")
print("=" * 70)
print(f"Working Directory: {working_dir}")
print(f"Test Directory:    {test_dir}")
print(f"WA Matrix:         {os.path.basename(path_to_WA)}")
print(f"Readout CSV:       {os.path.basename(readout)}")
print(f"Differentiate:     {differentiate}")
print(f"Diluting:          {diluting}")
print(f"Min Signal:        {min_signal}")
print(f"Output Directory:  {output_dir}")
print(f"Python Executable: {python_executable}")
print("=" * 70)

# Check if files exist
print("\nValidation:")
if os.path.exists(path_to_WA):
    print(f"✓ WA Matrix file found: {os.path.basename(path_to_WA)}")
else:
    print(f"✗ WA Matrix file NOT found: {os.path.basename(path_to_WA)}")
    print(f"  Looking in: {path_to_WA}")

if os.path.exists(readout):
    print(f"✓ Readout file found: {os.path.basename(readout)}")
else:
    print(f"✗ Readout file NOT found: {os.path.basename(readout)}")
    print(f"  Looking in: {readout}")

if os.path.exists(output_dir):
    print(f"✓ Output directory exists: {output_dir}")
else:
    print(f"✗ Output directory does NOT exist: {output_dir}")

print(f"✓ Python executable: {python_executable}")

print("=" * 70)

## Execute Decoder

Run the decoder with the configuration settings from the previous cell.


In [ ]:
# Use configured Python executable, fallback to system Python
exec_python = python_executable if os.path.exists(python_executable) else sys.executable

# Path to decode_N.py script (kept in project root)
script_path = os.path.join(working_dir, "decode_N.py")

if not os.path.exists(script_path):
    print(f"Error: decode_N.py not found at {script_path}")
    print(f"Expected location: {working_dir}")
else:
    # Build the decoder command
    cmd = [
        exec_python,
        script_path,
        "--path_to_WA", path_to_WA,
        "--readout", readout,
        "--differentiate", str(differentiate),
        "--diluting", "True" if diluting else "False"
    ]
    
    # Add optional min_signal parameter if provided
    if min_signal is not None:
        cmd.extend(["--min_signal", str(min_signal)])
    
    # Display the command being executed
    print("Executing decoder...")
    print("=" * 70)
    print(f"Command: {' '.join(cmd)}")
    print("=" * 70)
    print()
    
    # Run the decoder
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            cwd=output_dir,
            timeout=300  # 5-minute timeout
        )
        
        print(f"Return code: {result.returncode}")
        print()
        
        if result.stdout:
            print("STDOUT:")
            print(result.stdout)
        
        if result.stderr:
            print("STDERR:")
            print(result.stderr)
        
        # Check for success
        if result.returncode == 0:
            print("\n" + "=" * 70)
            print("✓ DECODING COMPLETED SUCCESSFULLY")
            print("=" * 70)
        else:
            print("\n" + "=" * 70)
            print("✗ DECODING FAILED")
            print("=" * 70)
    
    except subprocess.TimeoutExpired:
        print("✗ Decoder execution timed out (5 minutes)")
    except Exception as e:
        print(f"✗ Error running decoder: {e}")

## View Results

Display and analyze the decoded results from the previous execution.


In [ ]:
# Load and display the decoded results
output_csv = os.path.join(output_dir, "decoded_readouts.csv")

if os.path.exists(output_csv):
    print(f"Loading results from: {output_csv}")
    print("=" * 70)
    
    decoded_df = pd.read_csv(output_csv)
    
    print(f"\nDecoded Results (Total: {len(decoded_df)} readouts)")
    print("-" * 70)
    
    # Display the results in a formatted table
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_rows', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', None)
    
    print(decoded_df.to_string(index=False))
    
    print("\n" + "=" * 70)
    print(f"✓ Results file: {output_csv}")
    print("=" * 70)
    
    # Show summary statistics
    print("\nSummary:")
    print(f"  Total readouts: {len(decoded_df)}")
    print('You might need to scroll to the right to see all decoder results')
    if 'Decoded Type' in decoded_df.columns:
        print(f"\n  Decoded Types:")
        for dtype, count in decoded_df['Decoded Type'].value_counts().items():
            print(f"    - {dtype}: {count}")
    
else:
    print(f"✗ Results file not found: {output_csv}")
    print("The decoder may not have run successfully or the output path is incorrect.")
